# Session 0 — Setup, Sanity Check, and What Open Models Let Us See

**Series:** *Chasing "I Don't Know" in Modern Language Models*

**Session length:** ~30 min including download waits.
**Compute:** CPU is sufficient for this session. A GPU speeds things up but is not required.

## What this session does

This session doesn't teach concepts. Its job is to get your machine from *"clean laptop"* to *"the first cell of Session 1 runs on the first try."* Session 1 depends on you being able to load a language model and run a forward pass through it; here we install the packages, download the model, and verify everything works end-to-end with a small self-checking assertion.

We'll also set an **honest expectation** about what the diagnostic tools in this workshop can and cannot be pointed at, and specifically what part of the material transfers to commercial models like Claude and GPT — and what part doesn't.

---

## Part 1 — What you need

- **Python 3.11.** Some sessions use features that regress on 3.12 in the pinned `transformers` version. Check with `python --version`.
- **~10 GB of free disk.** Models cache to `~/.cache/huggingface/` by default.
- **Optional: a GPU with ≥6 GB VRAM.** Every session works CPU-only up to Session 6. Sessions 6–8 are noticeably faster with a GPU but a free Colab T4 also works.
- **A Hugging Face account** (free, ~2 minutes) is only needed for gated repositories such as Llama. Qwen and GPT-2 (the models this session uses) are **not gated** — you can skip HF login for now.

If you're on Colab: everything below runs in a free T4 runtime. Change nothing.

In [1]:
# Sanity check the environment
import sys
print(f"Python: {sys.version.split()[0]}")

if sys.version_info < (3, 11):
    print("  WARNING: this workshop is pinned to Python 3.11.")
    print("  Some cells may work on 3.10 or 3.12 but exact log-probability")
    print("  reproducibility is only guaranteed on 3.11.")
else:
    print("  Python version OK.")

Python: 3.11.15
  Python version OK.


## Part 2 — Install

From the notebooks directory:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

The pinned `requirements.txt` in this directory holds `torch==2.4.1`, `transformers==4.45.2`, and a handful of other packages at specific versions. Pinning matters because `transformers` and `tokenizers` occasionally change internal computations across minor versions in ways that break exact log-probability reproducibility. This workshop's numbers were verified against the pinned versions; if you upgrade, most things will still work but individual decimals may drift.

In [2]:
# Verify the key packages are installed at expected versions
##### UZYWAM NIECO INNEGO ZESTAWU: $conda activate chitchat
import importlib

for pkg, expected in [
    ("torch", "2.4"),
    ("transformers", "4.45"),
    ("tokenizers", "0.20"),
    ("accelerate", "1.0"),
    ("numpy", "1.26"),
    ("matplotlib", "3.9"),
]:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "?")
        match = "OK" if version.startswith(expected) else f"MISMATCH (expected {expected}.*)"
        print(f"  {pkg:15s} {version:12s} {match}")
    except ImportError:
        print(f"  {pkg:15s} NOT INSTALLED — run `pip install -r requirements.txt`")

  torch           2.14.0       MISMATCH (expected 2.4.*)
  transformers    5.14.1       MISMATCH (expected 4.45.*)
  tokenizers      0.22.2       MISMATCH (expected 0.20.*)
  accelerate      1.15.0       MISMATCH (expected 1.0.*)
  numpy           2.4.6        MISMATCH (expected 1.26.*)
  matplotlib      NOT INSTALLED — run `pip install -r requirements.txt`


## Part 3 — Which models, and why these specifically

The workshop's default is the matched pair **Qwen2.5-1.5B** (base) and **Qwen2.5-1.5B-Instruct**. Both are ~3 GB in bf16 and fit comfortably on modest hardware.

**Why Qwen2.5, not Qwen3?** Deliberate. Qwen3 is a newer generation with different training data and a different post-training recipe. Later sessions in this workshop reproduce, at small scale, findings from a study that used Qwen2.5 specifically. Staying on Qwen2.5 keeps the numerical comparisons in Sessions 6–8 honest. The *methods* would transfer to Qwen3 — only the specific numeric contrasts wouldn't.

**Why a matched base/instruct pair?** The workshop's central research angle in Sessions 6–8 is *what post-training does to a model*. That question only makes sense if you're comparing two checkpoints from the same pretraining run that differ *only* in whether post-training was applied. Cross-family comparisons (Qwen base vs. Llama instruct) can't tell you what post-training did — they mix too many variables. Same reason a controlled experiment uses a control group.

**GPT-2 small (124M)** is used only in Session 5, for one specific demo (the repetition loop) that modern instruction-tuned models resist. Kept around because its behavior teaches something that modern models no longer will.

**A note on pinned revisions.** In `workshop_utils.py`, each model's Hugging Face repository is pinned by a specific revision (a commit hash or the string `"main"`). Repositories can change under a stable name — tokenizer updates, config tweaks, safetensors reshards — so pinning by revision makes this workshop reproducible against a specific frozen state.

In [2]:
# Import the shared workshop helpers.
# We keep imports of `workshop_utils` explicit; the file lives alongside this
# notebook and grows session by session.
from workshop_utils import (
    load_model_and_tokenizer,
    pick_device,
    pick_dtype,
    set_all_seeds,
    PINNED_REVISIONS,
)

print("Pinned revisions:")
for repo, rev in PINNED_REVISIONS.items():
    print(f"  {repo:40s} @ {rev}")

device = pick_device()
dtype = pick_dtype(device)
print(f"\nDevice for this session: {device}")
print(f"Dtype:                    {dtype}")

Pinned revisions:
  Qwen/Qwen2.5-1.5B                        @ main
  Qwen/Qwen2.5-1.5B-Instruct               @ main
  openai-community/gpt2                    @ main
  microsoft/deberta-v3-base                @ main

Device for this session: cuda
Dtype:                    torch.bfloat16


## Part 4 — What "reproducibility" means here (and doesn't)

The workshop pins Python, package versions, model revisions, and RNG seeds. That gets us as close to reproducibility as is practically achievable. But there are limits, and it's worth stating them upfront so a curious reader isn't confused later when their numbers don't exactly match.

- **On the same machine, same seed, greedy decoding:** you should get identical token sequences and identical log-probabilities every run.

- **Across different GPU types** (e.g. T4 vs. RTX 3090 vs. Apple M-series MPS): the same seed will not produce bit-identical logits. Floating-point non-associativity in reduction operations means small numerical differences accumulate. The workshop's *conclusions* (e.g. "the top-1 token is the same," "the log-probability of the correct sentence exceeds the wrong one by ~X nats") are robust to this; the last few decimal places of any specific number are not.

- **CPU vs. GPU:** the same seeded computation on CPU vs. GPU is not bit-identical for the same reason.

- **Sampling (`temperature > 0`):** reproducible only if all RNG state is controlled. Our helper `set_all_seeds()` seeds Python's `random`, NumPy, PyTorch (both CPU and CUDA), and `PYTHONHASHSEED`. Where a demo depends on this, it's called explicitly.

Where a demo depends on exact numeric values, the reference device is stated in the text.

## Part 5 — Sanity check (the 10-minute run)

This is the load-bearing cell of Session 0. It downloads Qwen2.5-1.5B-Instruct if not already cached, loads it onto your device, runs a small deterministic generation, and asserts that the output matches an expected value.

If this cell finishes without error, **you are set up for the entire workshop**. If it fails, jump to Part 7 (troubleshooting) below.

On first run the model download may take several minutes depending on your connection. On subsequent runs the model is loaded from cache and this cell finishes in seconds.

In [3]:
import torch

set_all_seeds(0)

tokenizer, model = load_model_and_tokenizer(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device=device,
    dtype=dtype,
)

Loading Qwen/Qwen2.5-1.5B-Instruct @ main onto cuda as torch.bfloat16...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  loaded: 28 layers, hidden 1536, vocab 151936


Once loaded, let's run a small deterministic generation. The prompt is a plain factual completion. We'll use greedy decoding (`do_sample=False`) so the output is deterministic modulo the hardware caveats above.

In [4]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

set_all_seeds(0)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
    )

generated = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"Prompt:    {prompt!r}")
print(f"Completed: {generated!r}")

Prompt:    'The capital of France is'
Completed: ' Paris. The capital of Italy is Rome. What'


The completion should contain the word "Paris" (possibly with surrounding formatting or punctuation). Let's assert it, so this cell fails loudly if something has gone wrong:

In [5]:
assert "Paris" in generated, (
    f"Sanity check failed: expected 'Paris' in output, got {generated!r}. "
    "Something is wrong with your install — see the troubleshooting section below."
)
print("Sanity check passed.")

Sanity check passed.


One more diagnostic: peek at the model's log-probability for the correct answer. This previews the technique Session 3 will unpack fully.

In [6]:
# What did the model think about "Paris" as the next-token continuation of the prompt?
prompt_ids = tokenizer(prompt, return_tensors="pt").to(device).input_ids

with torch.no_grad():
    logits = model(prompt_ids).logits[0, -1, :]

log_probs = torch.log_softmax(logits.float(), dim=-1)

paris_id = tokenizer.encode(" Paris", add_special_tokens=False)[0]
paris_logprob = log_probs[paris_id].item()
paris_rank = int((log_probs > log_probs[paris_id]).sum().item())

print(f"Token ' Paris' has log-prob {paris_logprob:.3f} nats, "
      f"probability {torch.exp(torch.tensor(paris_logprob)).item():.3f}")
print(f"It is the model's #{paris_rank + 1} choice (rank 0 = top-1)")

top5_probs, top5_ids = torch.topk(log_probs.exp(), k=5)
print("\nTop 5 candidates for the next token:")
for i, (p, idx) in enumerate(zip(top5_probs.tolist(), top5_ids.tolist())):
    print(f"  {i+1}. {tokenizer.decode([idx])!r:20s} p={p:.4f}")

Token ' Paris' has log-prob -1.287 nats, probability 0.276
It is the model's #1 choice (rank 0 = top-1)

Top 5 candidates for the next token:
  1. ' Paris'             p=0.2760
  2. ' located'           p=0.0896
  3. ' a'                 p=0.0616
  4. ' ______'            p=0.0373
  5. ' the'               p=0.0373


**What this tells you.** The model has strong opinions about what comes after `"The capital of France is"`. The correct answer is almost certainly its top choice, with probability >90%. This is the pattern for a well-known fact. Session 3 will show you what happens on questions the model *doesn't* know — where the top choice is still confident but wrong.

---

## Part 6 — The reality box: open models vs. API models

This is the framing paragraph that recurs, in miniature, throughout the workshop:

> The demos in this workshop use *open-weight models* — models where the parameters are downloadable and every internal quantity (logits, attention weights, hidden states, sequence probabilities) can be inspected directly. This is what makes the diagnostic methods in Sessions 3, 5, and 6 possible.
>
> Commercial API models — Claude, GPT-4, Gemini — do not expose these internals. OpenAI's API returns top-20 token log-probabilities on request; Anthropic's returns none; Google's returns limited scoring information. This means that some of the specific instruments in this workshop (in particular, the sequence-probability ledger in Session 6) cannot be applied directly to models you access via API.
>
> **What still transfers:** the *mechanisms* are the same. A commercial model that fabricates confidently is, by all available evidence, doing so for structurally similar reasons to an open one. What you learn here about *why* hallucination is inevitable, what a semantic attractor is, what post-training does, and where spread-based detectors are blind, applies to Claude and GPT unchanged.
>
> **What each session's "So what, for a working scientist" box will do:** name, precisely and per session, the parts of the material that translate to API workflows and the parts that require open weights.

### So what, for a working scientist (Session 0 box)

You can complete every technical exercise in this workshop on a laptop or a free Colab. What you learn here is not a toy — it's the *lower bound* of what you can measure about a language model. If a demo shows a phenomenon on Qwen2.5-1.5B, it is at least as measurable on any larger open model; whether it is measurable on Claude or GPT depends on what those APIs expose, which is much less. The workshop teaches you what the ground truth looks like when you have full access, so you know what you're giving up when you don't.

## Part 7 — Troubleshooting

If the sanity check above failed, the four common failure modes are:

**1. Model download timed out or was interrupted.**
The Hugging Face client will resume on retry. Simply rerun the cell that calls `load_model_and_tokenizer`. If the resume itself is failing, delete the partial download from `~/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/` and retry.

**2. Insufficient disk space.**
Qwen2.5-1.5B-Instruct + Qwen2.5-1.5B (base, needed for Session 6) + GPT-2 small (Session 5) totals ~5 GB. If your home partition is small, set the `HF_HOME` environment variable to point at a larger disk *before* loading any model:
```python
import os
os.environ["HF_HOME"] = "/path/to/large/disk/hf_cache"
```

**3. torch/CUDA version mismatch.**
`RuntimeError: CUDA error: no kernel image is available for execution on the device` or similar. Your `torch` was built for a different CUDA version than your driver supports. Reinstall torch matching your CUDA version, per [PyTorch's install matrix](https://pytorch.org/get-started/locally/).

**4. Tokenizer version mismatch.**
`ValueError: Tokenizer class ... does not exist` or a segfault during tokenizer load. Reinstall via `pip install -r requirements.txt` in a clean venv.

If none of these match, print your `pip list` and share it along with the traceback.

## What you just did

- Verified your environment is at the pinned versions used across the workshop.
- Downloaded Qwen2.5-1.5B-Instruct and Ran a small deterministic generation with a passing assertion.
- Inspected the model's log-probability for the correct answer to a simple factual prompt — a preview of Session 3.
- Learned what "reproducibility" means in this workshop and what its limits are.
- Read the framing on open-weight vs. API models that will recur in a "So what" box in every session.

## Forward hook

You have a working language model and a tool for asking it questions. **Session 1** turns that tool into a diagnostic instrument: you'll watch a model generate text one token at a time, with visible top-*k* alternatives at every step, and build (in ~30 lines of Python) the core function that every subsequent session will reuse.